In [10]:
# =========================================================
# TF-IDF & VSM
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from src.tfidf.tfidf_process import build_tfidf

In [11]:
# =========================================================
# LOAD DATA HASIL PREPROCESSING
# =========================================================
base_dir = os.path.abspath('..')
csv_path = os.path.join(base_dir, 'data', 'cleaned_papers.csv')

df = pd.read_csv(csv_path)
df['cleaned_text'] = df['cleaned_text'].fillna('')

print(f'✅ Data loaded: {len(df)} baris')
df.head(3)

✅ Data loaded: 200 baris


,id,title,abstract,authors,year,source,category,cleaned_text
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,machine learning for microbiologists how to ev...
1,2,International conference on machine learning,"In this paper, we make the key delineation on ...","W Li, C Wang, G Cheng, Q Song",2023,Transactions on machine learning …,machine learning,international conference on machine learning i...
2,3,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,what is machine learning that one can employ i...


In [12]:
# =========================================================
# FIT TF-IDF (pakai src/tfidf/tfidf_process.py)
# =========================================================
save_dir = os.path.join(base_dir, 'data', 'tfidf')

print('⚙️ Menghitung TF-IDF...')
vectorizer, tfidf_matrix = build_tfidf(df['cleaned_text'].tolist(), save_dir)

print(f'✅ TF-IDF selesai!')
print(f'📊 Jumlah dokumen : {tfidf_matrix.shape[0]}')
print(f'📊 Jumlah term    : {tfidf_matrix.shape[1]}')

⚙️ Menghitung TF-IDF...
✅ TF-IDF selesai!
📊 Jumlah dokumen : 200
📊 Jumlah term    : 1722


In [16]:
# =========================================================
# REPRESENTASI VSM - TOP TERM PER DOKUMEN
# =========================================================
feature_names = vectorizer.get_feature_names_out()

print('\n=== Contoh Representasi VSM ===')
for i in range(min(3, len(df))):
    row = tfidf_matrix[i].toarray()[0]
    top_indices = row.argsort()[::-1][:10]
    top_terms = [(feature_names[j], round(row[j], 4)) for j in top_indices if row[j] > 0]
    print(f'\nDokumen {i+1}: {df["title"].iloc[i][:60]}')
    print(f'Top terms: {top_terms}')



=== Contoh Representasi VSM ===

Dokumen 1: Machine learning for microbiologists
Top terms: [('microbiologists', 0.4746), ('machine', 0.3508), ('work', 0.309), ('how', 0.2939), ('learning', 0.2436), ('grasp', 0.2373), ('learningbased', 0.2373), ('evaluate', 0.2112), ('topics', 0.2112), ('presented', 0.2112)]

Dokumen 2: International conference on machine learning
Top terms: [('blackbox', 0.2525), ('delineation', 0.2525), ('resolution', 0.2525), ('hierarchical', 0.2525), ('uncertainty', 0.2525), ('general', 0.2525), ('roles', 0.2525), ('banditsbased', 0.2525), ('conference', 0.2525), ('guiding', 0.2248)]

Dokumen 3: What is machine learning?
Top terms: [('learning', 0.4926), ('or', 0.2972), ('required', 0.2399), ('either', 0.2399), ('skills', 0.2399), ('types', 0.2135), ('automated', 0.2135), ('employ', 0.2135), ('unsupervised', 0.1981), ('human', 0.1981)]


In [17]:
# =========================================================
# VERIFIKASI RUMUS MANUAL (sesuai proposal)
# =========================================================
print('\n=== Verifikasi Rumus TF-IDF Manual ===')
sample_tokens = df['cleaned_text'].iloc[0].split()
sample_term = sample_tokens[0] if sample_tokens else ''
N = len(df)

tf = sample_tokens.count(sample_term) / len(sample_tokens)
df_term = sum(1 for text in df['cleaned_text'] if sample_term in text.split())
idf = np.log(N / df_term)
tfidf_manual = tf * idf

print(f'Term           : "{sample_term}"')
print(f'TF             : {sample_tokens.count(sample_term)} / {len(sample_tokens)} = {tf:.4f}')
print(f'IDF            : log({N} / {df_term}) = {idf:.4f}')
print(f'TF-IDF manual  : {tfidf_manual:.4f}')



=== Verifikasi Rumus TF-IDF Manual ===
Term           : "machine"
TF             : 4 / 31 = 0.1290
IDF            : log(200 / 53) = 1.3280
TF-IDF manual  : 0.1714


In [18]:
# =========================================================
# SIMPAN INDEX DOKUMEN
# =========================================================
base_dir = os.path.abspath('..')
index_path = os.path.join(base_dir, 'data', 'tfidf', 'doc_index.csv')

df[['id', 'title']].to_csv(index_path, index=False)
print(f'✅ Tersimpan: data/tfidf/doc_index.csv')
print(f'📊 Total: {len(df)} baris')
df[['id', 'title']].head(5)

✅ Tersimpan: data/tfidf/doc_index.csv
📊 Total: 200 baris


,id,title
0,1,Machine learning for microbiologists
1,2,International conference on machine learning
2,3,What is machine learning?
3,4,Amnesiac machine learning
4,5,Designing nanotheranostics with machine learning


In [19]:
# =========================================================
# SIMPAN SEMUA BOBOT TF-IDF KE CSV
# =========================================================

import pandas as pd

rows = []

# ambil semua nama term
feature_names = vectorizer.get_feature_names_out()

# loop seluruh dokumen
for doc_idx in range(len(df)):

    # ambil vector TF-IDF dokumen
    row = tfidf_matrix[doc_idx].toarray()[0]

    # loop seluruh term
    for term_idx, score in enumerate(row):

        # hanya simpan yang punya bobot
        if score > 0:

            rows.append({
                'doc_id': df.iloc[doc_idx]['id'],
                'title': df.iloc[doc_idx]['title'],
                'term': feature_names[term_idx],
                'tfidf_score': round(score, 4)
            })

# buat dataframe
tfidf_df = pd.DataFrame(rows)

# simpan csv
save_path = os.path.join(base_dir, 'data', 'tfidf', 'tfidf_weights.csv')

tfidf_df.to_csv(save_path, index=False)

print(f'✅ TF-IDF berhasil disimpan!')
print(f'📁 Lokasi: {save_path}')
print(f'📊 Total baris: {len(tfidf_df)}')

tfidf_df.head(20)

✅ TF-IDF berhasil disimpan!
📁 Lokasi: d:\Tugas Akhir\paperCi\data\tfidf\tfidf_weights.csv
📊 Total baris: 5229


,doc_id,title,term,tfidf_score
0,1,Machine learning for microbiologists,allow,0.2112
1,1,Machine learning for microbiologists,and,0.0478
2,1,Machine learning for microbiologists,evaluate,0.2112
3,1,Machine learning for microbiologists,for,0.0699
4,1,Machine learning for microbiologists,grasp,0.2373
5,1,Machine learning for microbiologists,how,0.2939
6,1,Machine learning for microbiologists,in,0.0563
7,1,Machine learning for microbiologists,learning,0.2436
8,1,Machine learning for microbiologists,learningbased,0.2373
9,1,Machine learning for microbiologists,machine,0.3508
